### EDA Pandas
##### Підготували Назар Михайлищук, Олена Яковенко та Оксана Москв'як

Мета цього дослідження: дізнатись детальнішу інформацію про локації міста Нешвілла, штат Теннессі, США, у яких трапилось найбільше порушень правил дорожнього руху та дослідити дані пов'язані із цими порушеннями.

In [14]:
import pandas as pd
import folium

In [15]:
data = pd.read_csv('tn_nashville_2020_04_01.csv')

C:\Users\partu\AppData\Local\Temp\ipykernel_11548\2804370486.py:1: DtypeWarning: Columns (6,8,15,16,17,22,23,24,25,29,30,31,32,33,35,36,37,38,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('tn_nashville_2020_04_01.csv')


#### Знаходимо локації у місті Нешвілл, в яких трапилось більше ніж 1500 порушень за увесь час
Повертаємо дані із початкової таблиці відфільтровані відповідно до того, чи є найпопулярніші локації у них.
 <br> Дані посортовано враховуючи найпопулярнішу локацію.

In [16]:
location_counts = data['location'].value_counts()[1:]
popular_locations = location_counts[location_counts >= 1500]
filtered_data = data[data['location'].isin(popular_locations.index)].reset_index(drop=True)

filtered_data.head()

,raw_row_number,date,time,location,lat,lng,precinct,reporting_area,zone,subject_age,...,raw_traffic_citation_issued,raw_misd_state_citation_issued,raw_suspect_ethnicity,raw_driver_searched,raw_passenger_searched,raw_search_consent,raw_search_arrest,raw_search_warrant,raw_search_inventory,raw_search_plain_view
0,232947,2010-10-10,NaN,"DOMINICAN DR & ROSA L PARKS BLVD, NASHVILLE, T...",36.187925,-86.798519,6.0,4403.0,611.0,27.0,...,False,NaN,N,False,False,False,False,False,False,False
1,232903,2010-10-10,10:05:00,"HAYWOOD LN & NOLENSVILLE PIKE, NASHVILLE, TN, ...",36.068208,-86.719133,8.0,8661.0,831.0,49.0,...,True,NaN,N,False,False,False,False,False,False,False
2,232905,2010-10-10,10:10:00,"BELL RD & BLUE HOLE RD, ANTIOCH, TN, 37013",36.045724,-86.674355,3.0,8719.0,331.0,31.0,...,False,NaN,N,False,False,False,False,False,False,False
3,232911,2010-10-10,10:18:00,"NOLENSVILLE PIKE & THOMPSON LN, NASHVILLE, TN,...",36.110927,-86.745188,NaN,8519.0,NaN,34.0,...,False,NaN,N,False,False,False,False,False,False,False
4,233227,2010-10-10,22:21:00,"APACHE TRL & HAYWOOD LN, NASHVILLE, TN, 37211",36.068404,-86.693941,3.0,8709.0,321.0,36.0,...,False,NaN,N,False,False,False,False,False,False,False


#### Знаходимо середній вік людей, які були зупинені поліцією у цій локації
Повертаємо таблицю, яка містить назву локації та середній вік людей, які потрапили у ДТП у даній локації.

In [17]:
mean_age_by_location = filtered_data.groupby('location')['subject_age'].mean().round().reset_index()
mean_age_by_location.columns = ['location', 'mean_age']

mean_age_by_location

,location,mean_age
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",39.0
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",42.0
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",37.0
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",37.0
4,"1101 BROADWAY, NASHVILLE, TN, 37203",34.0
...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",41.0
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",36.0
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",37.0
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",39.0


#### Знаходимо відсоток чоловіків та жінок, які були зупиненні поліцією у певній із локацій
Повертаємо таблицю, яка містить назву локації та відсотки відносно кількості <br>
людей чоловічої та жіночої статі, які порушили правила дорожнього руху в цій локації.

In [18]:
subject_sex_counts = filtered_data.groupby(['location', 'subject_sex']).size().reset_index(name='count')
total_counts = filtered_data.groupby('location').size().reset_index(name='all')
subject_sex_counts = subject_sex_counts.merge(total_counts, on='location')
subject_sex_counts['percentage'] = round((subject_sex_counts['count'] / subject_sex_counts['all'] * 100), 2)

result_table = subject_sex_counts.pivot_table(index='location', columns='subject_sex', values='percentage', fill_value=0).reset_index()
result_table.columns.name = None

result_table.columns = ['location', 'female', 'male']
result_table = result_table[['location', 'male', 'female']]

result_table.reset_index(drop=True)

,location,male,female
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",61.91,38.09
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",49.74,49.80
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",65.04,33.99
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",64.56,35.40
4,"1101 BROADWAY, NASHVILLE, TN, 37203",69.08,30.67
...,...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",49.49,50.46
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",57.10,42.30
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",60.99,38.60
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",50.83,49.03


#### Знаходимо дані реєстрації транспортного засобу, що скоїло порушення
Повертаємо таблицю, яка містить назву локації та відсотки відносно кількості <br>
транспортних засобів. які зареєстровані в штаті Tennessee (тобто штаті, де знаходиться місто Нешвілл), <br>
та в інших штатах, що скоїли порушення правил дорожнього руху в певній із локацій.

In [19]:
def is_from_tennessee(state):
    return state == 'TN'

state_of_car = filtered_data.groupby('location')['vehicle_registration_state'].apply(lambda x: x.apply(is_from_tennessee))
state_counts = state_of_car.groupby('location').value_counts().unstack(fill_value=0)
state_counts.columns = ['other', 'TN']

state_counts['all'] = state_counts['other'] + state_counts['TN']
state_counts['TN'] = round((state_counts['TN'] / state_counts['all']) * 100, 2)
state_counts['other states'] = round((100 - state_counts['TN']), 2)
state_counts = state_counts[['TN', 'other states']].reset_index()

state_counts.reset_index(drop=True)


,location,TN,other states
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",98.40,1.60
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",96.41,3.59
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",88.02,11.98
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",87.51,12.49
4,"1101 BROADWAY, NASHVILLE, TN, 37203",89.14,10.86
...,...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",98.92,1.08
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",92.28,7.72
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",95.26,4.74
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",98.54,1.46


#### Знаходимо годину, в яку трапилось найбільше порушень на цій локації
Повертаємо таблицю, яка містить назву локації та годину, в яку трапилось <br>
найбільше порушень правил дорожнього руху

In [20]:
filtered_data['time'].dropna()
filtered_data['hour'] = filtered_data['time'].str.split(':').str[0]

hour_counts = filtered_data.groupby(['location', 'hour']).size().reset_index(name='count')
most_popular_hour = hour_counts.loc[hour_counts.groupby('location')['count'].idxmax()]
most_popular_hour = most_popular_hour[['location', 'hour']]

most_popular_hour.reset_index(drop=True)


,location,hour
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",12
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",13
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",23
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",13
4,"1101 BROADWAY, NASHVILLE, TN, 37203",23
...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",10
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",19
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",02
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",13


#### Знаходимо найпопулярнішу причину зупинки 
Повертаємо таблицю, яка містить назву локації та найпопулярнішу причину зупинки.

In [21]:
reason_for_stop = filtered_data.groupby(['location', 'reason_for_stop']).size().reset_index(name='count')
most_popular_reason = reason_for_stop.loc[reason_for_stop.groupby('location')['count'].idxmax()]
most_popular_reason = most_popular_reason[['location', 'reason_for_stop']]

most_popular_reason.reset_index(drop=True)

,location,reason_for_stop
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",moving traffic violation
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",moving traffic violation
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",moving traffic violation
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",moving traffic violation
4,"1101 BROADWAY, NASHVILLE, TN, 37203",moving traffic violation
...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",moving traffic violation
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",moving traffic violation
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",vehicle equipment violation
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",moving traffic violation


#### Знаходимо найпопулярніше покарання
Повертаємо таблицю, яка містить назву локації та покарання, яке було видано найбільшу кількість разів за скоєння порушення правил дорожнього руху.

In [22]:
popular_outcome = filtered_data.groupby(['location', 'outcome']).size().reset_index(name='count')
most_popular_outcome = popular_outcome.loc[popular_outcome.groupby('location')['count'].idxmax()]
most_popular_outcome = most_popular_outcome[['location', 'outcome']]

most_popular_outcome.reset_index(drop=True)

,location,outcome
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",warning
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",warning
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",warning
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",warning
4,"1101 BROADWAY, NASHVILLE, TN, 37203",warning
...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",warning
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",warning
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",warning
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",warning


In [23]:
mean_age_by_location = mean_age_by_location.reset_index(drop=True)
result_table = result_table.reset_index(drop=True)
state_counts = state_counts.reset_index(drop=True)
most_popular_hour = most_popular_hour.reset_index(drop=True)
most_popular_reason = most_popular_reason.reset_index(drop=True)
most_popular_outcome = most_popular_outcome.reset_index(drop=True)


lat_lng = filtered_data[['location', 'lat', 'lng']].drop_duplicates().reset_index(drop=True)

final_table = mean_age_by_location.merge(state_counts, on='location', how='left')
final_table = final_table.merge(result_table, on='location', how='left')
final_table = final_table.merge(most_popular_hour, on='location', how='left')
final_table = final_table.merge(most_popular_reason, on='location', how='left')
final_table = final_table.merge(most_popular_outcome, on='location', how='left')
final_table = final_table.merge(lat_lng, on='location', how='left')

final_table


,location,mean_age,TN,other states,male,female,hour,reason_for_stop,outcome,lat,lng
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",39.0,98.40,1.60,61.91,38.09,12,moving traffic violation,warning,36.098636,-86.712964
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",42.0,96.41,3.59,49.74,49.80,13,moving traffic violation,warning,36.183116,-86.595200
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",37.0,88.02,11.98,65.04,33.99,23,moving traffic violation,warning,36.157995,-86.784179
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",37.0,87.51,12.49,64.56,35.40,13,moving traffic violation,warning,36.157878,-86.784436
4,"1101 BROADWAY, NASHVILLE, TN, 37203",34.0,89.14,10.86,69.08,30.67,23,moving traffic violation,warning,36.156484,-86.786737
...,...,...,...,...,...,...,...,...,...,...,...
306,"S GRAYCROFT AVE & CHADWELL DR, MADISON, TN, 37115",41.0,98.92,1.08,49.49,50.46,10,moving traffic violation,warning,36.254854,-86.733584
307,"SIMMONS AVE & THOMPSON LN, NASHVILLE, TN, 37211",36.0,92.28,7.72,57.10,42.30,19,moving traffic violation,warning,36.110615,-86.740699
308,"W TRINITY LN & BRICK CHURCH PIKE, NASHVILLE, T...",37.0,95.26,4.74,60.99,38.60,02,vehicle equipment violation,warning,36.206678,-86.777962
309,"W TRINITY LN & FREE SILVER RD, NASHVILLE, TN, ...",39.0,98.54,1.46,50.83,49.03,13,moving traffic violation,warning,36.207057,-86.796000


In [27]:
final_table = final_table.dropna(subset=['lat', 'lng'])
unique_locations = final_table.groupby(['lat', 'lng']).first().reset_index()

m = folium.Map(location=[unique_locations['lat'].mean(), unique_locations['lng'].mean()], zoom_start=12)

for index, row in unique_locations.iterrows():
    html = f"""
    <div style="font-family: Arial, sans-serif; border-radius: 8px; padding: 10px; box-shadow: 2px 2px 5px #888888; background-color: #f9f9f9;">
        <h4 style="margin-bottom: 10px; color: #2c3e50;">Location Information</h4>
        <p style="color: #e74c3c;"><strong>Location:</strong> {row['location']}</p>
        <p style="color: #3498db;"><strong>Mean Age:</strong> {row['mean_age']}</p>
        <p style="color: #2ecc71;"><strong>From TN:</strong> {row['TN']}</p>
        <p style="color: #f39c12;"><strong>Other States:</strong> {row['other states']}</p>
        <p style="color: #9b59b6;"><strong>Male:</strong> {row['male']}%</p>
        <p style="color: #e67e22;"><strong>Female:</strong> {row['female']}%</p>
        <p style="color: #1abc9c;"><strong>Hour:</strong> {row['hour']}</p>
        <p style="color: #d35400;"><strong>Reason for Stop:</strong> {row['reason_for_stop']}</p>
        <p style="color: #c0392b;"><strong>Outcome:</strong> {row['outcome']}</p>
    </div>
    """
    folium.Marker(
        [row['lat'], row['lng']], 
        popup=folium.Popup(html, max_width=300)
    ).add_to(m)

m.save('nashville_map.html')